In [0]:
print(1)

In [0]:
%pip install --quiet pymupdf FlagEmbedding graphrag
dbutils.library.restartPython()


In [0]:
# ==== 設定 ====
# 対象PDFが置いてあるUnity Catalogボリューム
volume_path = "dbfs:/Volumes/koiso_databricks_04_2991952537520556/default/vol/rag/"

# 保存先（Unity Catalog）
catalog_name = "koiso_databricks_04_2991952537520556"
schema_name  = "default"
table_chunks = f"{catalog_name}.{schema_name}.rag_chunks"   # チャンク化テキストの表
table_index  = f"{catalog_name}.{schema_name}.rag_index"    # ベクトル付きの表（最終）

# チャンク設定（文字ベースで素直に分割：LLM呼び出しなし前提）
chunk_size   = 2000   # 1チャンクあたりの文字数
chunk_overlap= 200    # チャンク間のオーバーラップ

# 埋め込み利用優先度：
# 1) Databricksの組込み 'system.ai.bge_m3' が使える場合はそれを使う
# 2) 使えない場合は FlagEmbedding(BGE-M3) にフォールバック
use_system_bge_m3 = True


In [0]:
import io
import fitz
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# PDFバイナリのDataFrameを取得
pdf_df = (spark.read.format("binaryFile")
          .option("pathGlobFilter", "*.pdf")
          .load(volume_path)
          .select("path", "modificationTime", "length", "content"))

# ページ単位でテキスト抽出 → チャンク化
# 関数は禁止のため、そのままノート上で逐次処理
rows = []
for r in pdf_df.toLocalIterator():  # メモリ負荷を抑えつつ順次処理
    path = r["path"]
    content = r["content"]  # BinaryType
    doc = fitz.open(stream=io.BytesIO(content), filetype="pdf")
    print(f"Loaded: {path} (pages={doc.page_count})")

    for page_idx in range(doc.page_count):
        page = doc.load_page(page_idx)
        text = page.get_text("text") or ""
        text = text.strip()
        if not text:
            continue

        # 文字ベースでチャンク化（オーバーラップつき）
        start = 0
        chunk_id = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk_text = text[start:end]
            rows.append(Row(
                source_path = path,
                file_name   = path.split("/")[-1],
                page        = int(page_idx + 1),
                chunk_id    = int(chunk_id),
                chunk_text  = chunk_text
            ))
            chunk_id += 1
            if end == len(text):
                break
            start = end - chunk_overlap  # オーバーラップ分戻す。負にならないようにする
            if start < 0:
                start = 0

# チャンクテーブル（テキストのみ）をDeltaとして作成
schema = StructType([
    StructField("source_path", StringType(), False),
    StructField("file_name",   StringType(), False),
    StructField("page",        IntegerType(), False),
    StructField("chunk_id",    IntegerType(), False),
    StructField("chunk_text",  StringType(), False),
])

df_chunks = spark.createDataFrame(rows, schema=schema)
df_chunks.write.mode("overwrite").format("delta").saveAsTable(table_chunks)

print(f"チャンク表を作成: {table_chunks} / 件数={df_chunks.count()}")
display(spark.table(table_chunks).limit(5))


In [0]:
from FlagEmbedding import BGEM3FlagModel
from pyspark.sql.types import ArrayType, FloatType

if not use_system_bge_m3:
    print("既にフォールバック利用モードです。")

# 上のテストで失敗していた場合はここで自動フォールバック
if use_system_bge_m3:
    print("上のセルで system.ai.bge_m3 の利用に失敗していた場合のみ、このセルを実行してください。")

# チャンクを取り出し → Pythonで埋め込み計算 → 再度DataFrameへ
dfc = spark.table(table_chunks).orderBy("file_name", "page", "chunk_id")
rows = []
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

for r in dfc.toLocalIterator():
    text = r["chunk_text"]
    emb = model.encode([text], batch_size=1, max_length=8192)
    dense = emb["dense_vecs"][0]
    rows.append((
        r["source_path"],
        r["file_name"],
        int(r["page"]),
        int(r["chunk_id"]),
        text,
        [float(x) for x in dense]
    ))

schema = StructType([
    StructField("source_path", StringType(), False),
    StructField("file_name",   StringType(), False),
    StructField("page",        IntegerType(), False),
    StructField("chunk_id",    IntegerType(), False),
    StructField("text",        StringType(), False),
    StructField("vector",      ArrayType(FloatType()), False),
])

df_index = spark.createDataFrame(rows, schema=schema)
df_index.write.mode("overwrite").format("delta").saveAsTable(table_index)

print(f"[Fallback: BAAI/bge-m3] ベクトル付き表を作成: {table_index} / 件数={df_index.count()}")
display(spark.table(table_index).limit(5))
